In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_validate,
    GridSearchCV
)

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score
)

clf_df = pd.read_csv("data/classification_escalation.csv")

display(clf_df.head())

In [3]:
print("Dataset shape:")
print(clf_df.shape)

print("\nColumn names:")
print(clf_df.columns.tolist())

print("\nData types:")
print(clf_df.dtypes)

print("\nMissing values:")
print(clf_df.isnull().sum())

print("\nStatistical summary:")
display(clf_df.describe(include="all"))

Dataset shape:
(900, 7)

Column names:
['ticket_id', 'complexity_score', 'team_experience_years', 'priority', 'channel', 'customer_tenure_months', 'escalated']

Data types:
ticket_id                     str
complexity_score          float64
team_experience_years     float64
priority                      str
channel                       str
customer_tenure_months    float64
escalated                   int64
dtype: object

Missing values:
ticket_id                  0
complexity_score          22
team_experience_years     36
priority                  13
channel                    9
customer_tenure_months    31
escalated                  0
dtype: int64

Statistical summary:


,ticket_id,complexity_score,team_experience_years,priority,channel,customer_tenure_months,escalated
count,900,878.000000,864.000000,887,891,869.000000,900.000000
unique,900,NaN,NaN,4,4,NaN,NaN
top,T0000,NaN,NaN,Medium,Portal,NaN,NaN
freq,1,NaN,NaN,348,245,NaN,NaN
mean,NaN,5.523918,7.482639,NaN,NaN,40.018745,0.212222
std,NaN,2.869451,4.528235,NaN,NaN,26.126088,0.409109
min,NaN,1.000000,0.000000,NaN,NaN,1.065286,0.000000
25%,NaN,3.000000,4.000000,NaN,NaN,20.085430,0.000000
50%,NaN,5.000000,8.000000,NaN,NaN,34.649084,0.000000
75%,NaN,8.000000,11.000000,NaN,NaN,54.269475,0.000000


In [4]:
print("Target distribution:")
print(clf_df["escalated"].value_counts())

print("\nTarget percentages:")
print(clf_df["escalated"].value_counts(normalize=True) * 100)

print("\nUnique target values:")
print(clf_df["escalated"].unique())

Target distribution:
escalated
0    709
1    191
Name: count, dtype: int64

Target percentages:
escalated
0    78.777778
1    21.222222
Name: proportion, dtype: float64

Unique target values:
[0 1]


In [5]:
X_clf = clf_df.drop(columns=["ticket_id", "escalated"])

y_clf = clf_df["escalated"]


# Identify numerical and categorical features

numeric_features = X_clf.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_features = X_clf.select_dtypes(
    include=["object", "category", "str"]
).columns.tolist()


print("Numerical features:", numeric_features)
print("Categorical features:", categorical_features)

print("\nFeature shape:", X_clf.shape)
print("Target shape:", y_clf.shape)

Numerical features: ['complexity_score', 'team_experience_years', 'customer_tenure_months']
Categorical features: ['priority', 'channel']

Feature shape: (900, 5)
Target shape: (900,)


In [6]:
X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
    X_clf,
    y_clf,
    test_size=0.2,
    random_state=42,
    stratify=y_clf
)


print("Training features:", X_train_clf.shape)
print("Testing features:", X_test_clf.shape)

print("\nTraining target distribution:")
print(y_train_clf.value_counts(normalize=True) * 100)

print("\nTesting target distribution:")
print(y_test_clf.value_counts(normalize=True) * 100)

Training features: (720, 5)
Testing features: (180, 5)

Training target distribution:
escalated
0    78.75
1    21.25
Name: proportion, dtype: float64

Testing target distribution:
escalated
0    78.888889
1    21.111111
Name: proportion, dtype: float64


In [7]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

In [8]:
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

In [9]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

In [10]:
logistic_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000, random_state=42))
])

knn_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", KNeighborsClassifier())
])

tree_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", DecisionTreeClassifier(random_state=42))
])

print("Three classification pipelines created successfully.")

Three classification pipelines created successfully.


In [11]:
cv_strategy = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

print("5-fold stratified cross-validation created successfully.")

5-fold stratified cross-validation created successfully.


In [12]:
models = {
    "Logistic Regression": logistic_pipeline,
    "K-Nearest Neighbors": knn_pipeline,
    "Decision Tree": tree_pipeline
}


scoring = {
    "f1": "f1",
    "precision": "precision",
    "recall": "recall"
}

cv_results = []

for model_name, model in models.items():

    scores = cross_validate(
        model,
        X_train_clf,
        y_train_clf,
        cv=cv_strategy,
        scoring=scoring,
        n_jobs=-1
    )

    cv_results.append({
        "Model": model_name,
        "Mean F1": scores["test_f1"].mean(),
        "Std F1": scores["test_f1"].std(),
        "Mean Precision": scores["test_precision"].mean(),
        "Mean Recall": scores["test_recall"].mean()
    })

cv_results_df = pd.DataFrame(cv_results)

display(cv_results_df.round(4))

,Model,Mean F1,Std F1,Mean Precision,Mean Recall
0,Logistic Regression,0.3927,0.0627,0.6240,0.2882
1,K-Nearest Neighbors,0.3608,0.0503,0.5617,0.2677
2,Decision Tree,0.3307,0.0581,0.3183,0.3467


In [13]:


param_grid = {
    "classifier__C": [0.01, 0.1, 1, 10, 100],
    "classifier__class_weight": [None, "balanced"]
}


grid_search = GridSearchCV(
    estimator=logistic_pipeline,
    param_grid=param_grid,
    scoring="f1",
    cv=cv_strategy,
    n_jobs=-1,
    verbose=1
)


grid_search.fit(X_train_clf, y_train_clf)


print("Best parameters:")
print(grid_search.best_params_)

print("\nBest cross-validation F1-score:")
print(round(grid_search.best_score_, 4))

Fitting 5 folds for each of 10 candidates, totalling 50 fits
Best parameters:
{'classifier__C': 0.1, 'classifier__class_weight': 'balanced'}

Best cross-validation F1-score:
0.52


In [14]:
best_clf_model = grid_search.best_estimator_


y_pred_clf = best_clf_model.predict(X_test_clf)


print("Classification Report:")
print(classification_report(y_test_clf, y_pred_clf))


print("\nConfusion Matrix:")
print(confusion_matrix(y_test_clf, y_pred_clf))


test_f1 = f1_score(y_test_clf, y_pred_clf)

print("\nFinal Test F1-score:", round(test_f1, 4))

Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.82      0.87       142
           1       0.52      0.71      0.60        38

    accuracy                           0.80       180
   macro avg       0.72      0.77      0.73       180
weighted avg       0.83      0.80      0.81       180


Confusion Matrix:
[[117  25]
 [ 11  27]]

Final Test F1-score: 0.6


Experiment B

In [15]:


from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("Regression libraries imported successfully.")

Regression libraries imported successfully.


In [16]:


reg_df = pd.read_csv("data/resolution_time_regression.csv")

display(reg_df.head())

,ticket_id,complexity_score,team_experience_years,priority,channel,customer_tenure_months,resolution_time_hours
0,R0000,7.0,10.0,Medium,Chat,32.638377,8.38724
1,R0001,4.0,15.0,Medium,Chat,45.273760,0.50000
2,R0002,4.0,13.0,Low,Portal,44.878323,0.50000
3,R0003,1.0,10.0,High,Phone,40.095034,0.50000
4,R0004,3.0,15.0,Low,Portal,47.451134,0.50000


In [17]:


print("Dataset shape:")
print(reg_df.shape)

print("\nColumn names:")
print(reg_df.columns.tolist())

print("\nData types:")
print(reg_df.dtypes)

print("\nMissing values:")
print(reg_df.isnull().sum())

print("\nStatistical summary:")
display(reg_df.describe(include="all"))

Dataset shape:
(800, 7)

Column names:
['ticket_id', 'complexity_score', 'team_experience_years', 'priority', 'channel', 'customer_tenure_months', 'resolution_time_hours']

Data types:
ticket_id                     str
complexity_score          float64
team_experience_years     float64
priority                      str
channel                       str
customer_tenure_months    float64
resolution_time_hours     float64
dtype: object

Missing values:
ticket_id                  0
complexity_score          16
team_experience_years     24
priority                   0
channel                    0
customer_tenure_months    20
resolution_time_hours      0
dtype: int64

Statistical summary:


,ticket_id,complexity_score,team_experience_years,priority,channel,customer_tenure_months,resolution_time_hours
count,800,784.000000,776.000000,800,800,780.000000,800.000000
unique,800,NaN,NaN,4,4,NaN,NaN
top,R0000,NaN,NaN,Medium,Phone,NaN,NaN
freq,1,NaN,NaN,311,206,NaN,NaN
mean,NaN,5.545918,7.496134,NaN,NaN,39.577416,5.873160
std,NaN,2.849200,4.479342,NaN,NaN,26.510500,4.144974
min,NaN,1.000000,0.000000,NaN,NaN,0.884553,0.500000
25%,NaN,3.000000,4.000000,NaN,NaN,19.697823,2.631319
50%,NaN,6.000000,8.000000,NaN,NaN,33.979577,5.387386
75%,NaN,8.000000,11.000000,NaN,NaN,54.299824,8.345903


In [18]:


print("Resolution time summary:")
display(reg_df["resolution_time_hours"].describe())

print("\nNumber of unique target values:")
print(reg_df["resolution_time_hours"].nunique())

Resolution time summary:


count    800.000000
mean       5.873160
std        4.144974
min        0.500000
25%        2.631319
50%        5.387386
75%        8.345903
max       27.488229
Name: resolution_time_hours, dtype: float64


Number of unique target values:
729


In [19]:


X_reg = reg_df.drop(
    columns=["ticket_id", "resolution_time_hours"]
)

y_reg = reg_df["resolution_time_hours"]


numeric_features_reg = X_reg.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_features_reg = X_reg.select_dtypes(
    include=["object", "category", "str"]
).columns.tolist()

print("Numerical features:", numeric_features_reg)
print("Categorical features:", categorical_features_reg)

print("\nFeature shape:", X_reg.shape)
print("Target shape:", y_reg.shape)

Numerical features: ['complexity_score', 'team_experience_years', 'customer_tenure_months']
Categorical features: ['priority', 'channel']

Feature shape: (800, 5)
Target shape: (800,)


In [20]:


X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg,
    y_reg,
    test_size=0.2,
    random_state=42
)

print("Training features:", X_train_reg.shape)
print("Testing features:", X_test_reg.shape)

print("\nTraining target:", y_train_reg.shape)
print("Testing target:", y_test_reg.shape)

print("\nTraining target summary:")
print(y_train_reg.describe())

print("\nTesting target summary:")
print(y_test_reg.describe())

Training features: (640, 5)
Testing features: (160, 5)

Training target: (640,)
Testing target: (160,)

Training target summary:
count    640.000000
mean       5.766726
std        4.094980
min        0.500000
25%        2.574068
50%        5.337467
75%        8.155341
max       26.615113
Name: resolution_time_hours, dtype: float64

Testing target summary:
count    160.000000
mean       6.298896
std        4.326252
min        0.500000
25%        3.363404
50%        5.629909
75%        8.747511
max       27.488229
Name: resolution_time_hours, dtype: float64


In [21]:
numeric_transformer_reg = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

In [22]:
categorical_transformer_reg = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

In [23]:
preprocessor_reg = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer_reg, numeric_features_reg),
        ("cat", categorical_transformer_reg, categorical_features_reg)
    ]
)

print("Regression preprocessing created successfully.")

Regression preprocessing created successfully.


In [24]:
linear_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor_reg),
    ("regressor", LinearRegression())
])

ridge_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor_reg),
    ("regressor", Ridge())
])

tree_reg_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor_reg),
    ("regressor", DecisionTreeRegressor(random_state=42))
])

print("Three regression pipelines created successfully.")

Three regression pipelines created successfully.


In [25]:
cv_strategy_reg = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

print("5-fold regression cross-validation created successfully.")

5-fold regression cross-validation created successfully.


In [26]:
reg_models = {
    "Linear Regression": linear_pipeline,
    "Ridge Regression": ridge_pipeline,
    "Decision Tree Regressor": tree_reg_pipeline
}

reg_cv_results = []

for model_name, model in reg_models.items():

    scores = cross_validate(
        model,
        X_train_reg,
        y_train_reg,
        cv=cv_strategy_reg,
        scoring="neg_mean_absolute_error",
        n_jobs=-1
    )

    mae_scores = -scores["test_score"]

    reg_cv_results.append({
        "Model": model_name,
        "Mean MAE": mae_scores.mean(),
        "Std MAE": mae_scores.std()
    })

reg_cv_results_df = pd.DataFrame(reg_cv_results)

display(reg_cv_results_df.round(4))

,Model,Mean MAE,Std MAE
0,Linear Regression,2.0351,0.1717
1,Ridge Regression,2.0351,0.1714
2,Decision Tree Regressor,3.2391,0.0538


In [27]:
param_grid_reg = {
    "regressor__alpha": [0.001, 0.01, 0.1, 1, 10, 100, 1000]
}


ridge_grid_search = GridSearchCV(
    estimator=ridge_pipeline,
    param_grid=param_grid_reg,
    scoring="neg_mean_absolute_error",
    cv=cv_strategy_reg,
    n_jobs=-1,
    verbose=1
)


ridge_grid_search.fit(X_train_reg, y_train_reg)


print("Best parameters:")
print(ridge_grid_search.best_params_)

print("\nBest cross-validation MAE:")
print(round(-ridge_grid_search.best_score_, 4))

Fitting 5 folds for each of 7 candidates, totalling 35 fits
Best parameters:
{'regressor__alpha': 0.1}

Best cross-validation MAE:
2.0351


In [28]:


best_reg_model = ridge_grid_search.best_estimator_


y_pred_reg = best_reg_model.predict(X_test_reg)


test_mae = mean_absolute_error(y_test_reg, y_pred_reg)

test_rmse = np.sqrt(mean_squared_error(y_test_reg, y_pred_reg))

test_r2 = r2_score(y_test_reg, y_pred_reg)


print("Final Regression Test Results:")

print("MAE:", round(test_mae, 4))

print("RMSE:", round(test_rmse, 4))

print("R-squared:", round(test_r2, 4))

Final Regression Test Results:
MAE: 2.2301
RMSE: 3.4028
R-squared: 0.3775


Report
1. Why stratify the classification split?

Stratification helps maintain approximately the same proportion of each class in both the training and testing sets. In my classification dataset, approximately 79% of the tickets were not escalated, while 21% were escalated. By using stratification, I ensured that both sets represented the original class distribution, making the model evaluation more reliable.
   
2. Why must preprocessing be inside the pipeline?
Preprocessing should be inside the pipeline to prevent data leakage. If I handle missing values or scale the data using the entire dataset before splitting it into training and validation folds, information from the validation data could influence the training process. By keeping preprocessing inside the pipeline, each fold learns the required transformations only from its training data, making the evaluation fairer and more reliable.
 
3. What does fold variability tell you?
Fold variability tells us how consistently a model performs across different data splits during cross-validation. A lower standard deviation means the model performs more consistently, while a higher standard deviation means its performance changes more between folds. Therefore, I considered both the average performance and standard deviation when comparing my models.

4. Why should the test set be evaluated only once?
The test set should be evaluated only once because it represents completely unseen data. If I repeatedly check the test results and change my model based on them, I may unintentionally adapt the model to that particular test set. This would make the final evaluation less reliable. Therefore, I used cross-validation and training data for model selection and tuning, keeping the test set for final evaluation.

5. What deployment constraints might change your choice?

   In a real-world environment, the best model may depend on more than its prediction performance. Factors such as prediction speed, available computer resources, memory, ease of understanding the predictions, and maintenance requirements could influence the choice.

For example, in my classification experiment, identifying as many genuinely escalated tickets as possible is important because missing an urgent ticket could delay customer support. In my regression experiment, the average prediction error of approximately 2.23 hours must be acceptable for the company's planning needs. Therefore, the final model should satisfy both technical requirements and real-world business needs.